In [13]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial.distance import euclidean
from sklearn.base import BaseEstimator, ClusterMixin, TransformerMixin
from sklearn.cluster import DBSCAN, KMeans, MiniBatchKMeans
from sklearn.compose import ColumnTransformer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

import pyspark.sql.functions as f
from pyspark.sql import SparkSession
from pyspark import SparkConf

import tqdm

In [2]:
parameters = {
    "spark.driver.maxResultSize": "3g",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.execution.arrow.pyspark.enabled": True,

    # https://docs.kedro.org/en/stable/integrations/pyspark_integration.html#tips-for-maximising-concurrency-using-threadrunner
    "spark.scheduler.mode": "FAIR",
    "spark.driver.extraJavaOptions": "-Djava.security.manager=allow",
    "spark.executor.extraJavaOptions": "-Djava.security.manager=allow",

    "spark.sql.legacy.parquet.nanosAsLong": True,
    "spark.driver.memory": "40g",
}

spark_conf = SparkConf().setAll(parameters.items())
spark = SparkSession.builder.appName('New').enableHiveSupport().config(conf=spark_conf).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/02 12:20:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/06/02 12:20:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/06/02 12:20:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [14]:
ftr_transactions = spark.read.parquet("../data/04_feature/ftr_transactions")

In [15]:
ftr_transactions.filter(
    f.col("is_loyal") == 1
).filter(
    f.col("_observ_end_dt") == "2024-12-31"
).show(20, truncate=False)

+---------------------+--------------+------------------+------------------+------------------+------------------+----------------------------+-------------------------+--------------------------+------------------------+-----------------------+---------------------------------+---------------------------+-------------------------------+------------------+------------------------+---------------------------+------------------------------+----------------------------+----------------------------------------+-------------------------------------+--------------------------------------+----------------------------------+-------------------------------------+-----------------------------------+-----------------------------------------------+--------------------------------------------+---------------------------------------------+-----------------------------------+--------------------------------------+------------------------------------+------------------------------------------------+--

In [17]:
ftr_transactions.filter(
    f.col("_id") == "7058KRR__966561214255"
).orderBy(
    "_id", f.desc("_observ_end_dt")
).select(
    "_id",
    "_observ_end_dt",
    "month_net_sales",
    "month_avg_order",
    "month_avg_order_mean_past_5_next_0_months",
    "month_avg_order_mean_past_11_next_0_months",
    "month_avg_order_mean_past_23_next_0_months",
).show(30, truncate=False)

+---------------------+--------------+------------------+------------------+-----------------------------------------+------------------------------------------+------------------------------------------+
|_id                  |_observ_end_dt|month_net_sales   |month_avg_order   |month_avg_order_mean_past_5_next_0_months|month_avg_order_mean_past_11_next_0_months|month_avg_order_mean_past_23_next_0_months|
+---------------------+--------------+------------------+------------------+-----------------------------------------+------------------------------------------+------------------------------------------+
|7058KRR__966561214255|2025-06-30    |0.0               |0.0               |0.0                                      |41.13750012715658                         |62.103186173872515                        |
|7058KRR__966561214255|2025-05-31    |0.0               |0.0               |17.738333384195965                       |41.13750012715658                         |65.06048075358073  

In [5]:
ftr_transactions.withColumn(
    "loyalty_segment",
    f.when(
        f.col("is_loyal") == 1,
        "loyal"
    ).when(
        f.col("is_potential_loyal") == 1,
        "potential_loyal"
    ).when(
        f.col("is_uncommited") == 1,
        "uncommited"
    ).when(
        f.col("is_new_joiner") == 1,
        "new_joiner"
    ).when(
        f.col("is_lost") == 1,
        "lost"
    ).when(
        f.col("is_gone") == 1,
        "gone"
    ).otherwise("uncommited")
).filter(
    f.col("_observ_end_dt") == "2024-12-31"
).groupBy("loyalty_segment").agg(
    f.countDistinct("_id")
).orderBy("loyalty_segment").show()

+---------------+-------------------+
|loyalty_segment|count(DISTINCT _id)|
+---------------+-------------------+
|           gone|            1078619|
|           lost|             628178|
|          loyal|             184702|
|     new_joiner|             305103|
|potential_loyal|             120279|
|     uncommited|             643399|
+---------------+-------------------+



In [6]:
segmentation_df = spark.read.parquet("../data/07_model_output/segmentation/transactions/segmentation_model_output.parquet")

In [7]:
segmentation_df.filter(
    f.col("_observ_end_dt") >= "2021-01-01"
).filter(
    f.col("_observ_end_dt") < "2025-01-01"
).filter(
    f.col("is_active") > 0
).groupBy("loyalty_segment", "cluster_id").agg(
    f.countDistinct("_id")
).orderBy("loyalty_segment", "cluster_id").show()

+---------------+----------+-------------------+
|loyalty_segment|cluster_id|count(DISTINCT _id)|
+---------------+----------+-------------------+
|          loyal|         0|             380769|
|          loyal|         1|             243120|
|          loyal|         2|              48480|
|          loyal|         3|             118293|
|     new_joiner|         0|             407146|
|     new_joiner|         1|             157500|
|     new_joiner|         2|             431724|
|     new_joiner|         3|              69607|
|     new_joiner|         4|             186528|
|potential_loyal|         0|             318084|
|potential_loyal|         1|             203417|
|potential_loyal|         2|             104180|
|     uncommited|         0|             145326|
|     uncommited|         1|              51385|
|     uncommited|         2|             160859|
|     uncommited|         3|             143835|
|     uncommited|         4|              73136|
|     uncommited|   

In [ ]:
agg_cols = [
    f.round(f.mean("most_trx_branch_n_places_within_0300m__All", 2).alias("most_trx_branch_n_places_within_0300m__All")),
    f.round(f.mean("most_trx_branch_pop_density", 2).alias("most_trx_branch_pop_density")),
    f.round(f.mode("most_frequent_day_of_month_mean_past_11_next_0_months", 2).alias("most_frequent_day_of_month_mean_past_11_next_0_months")),
    f.round(f.mode("most_frequent_day_of_week_mean_past_11_next_0_months", 2).alias("most_frequent_day_of_week_mean_past_11_next_0_months")),
    f.round(f.median("last_trx_net_sales", 2).alias("last_trx_net_sales")),
    f.round(f.median("last_trx_total_discount_percentual", 2).alias("last_trx_total_discount_percentual")),
    f.round(f.median("last_trx_current_mileage", 2).alias("last_trx_current_mileage")),
    f.round(f.median("month_avg_mileage_per_day_nullif_past_11_next_0_months", 2).alias("month_avg_mileage_per_day_nullif_past_11_next_0_months")),
    f.round(f.mean("customer_synthetic_oil", 2).alias("customer_synthetic_oil")),
    f.round(f.median("month_avg_order_mean_past_11_next_0_months", 2).alias("month_avg_order_mean_past_11_next_0_months")),
    f.round(f.median("month_avg_order_max_past_11_next_0_months", 2).alias("month_avg_order_max_past_11_next_0_months")),
    f.round(f.median("month_avg_percent_discount_mean_past_11_next_0_months", 2).alias("month_avg_percent_discount_mean_past_11_next_0_months")),
    f.round(f.median("month_avg_percent_discount_max_past_11_next_0_months", 2).alias("month_avg_percent_discount_max_past_11_next_0_months")),
    f.round(f.median("is_active_sum_past_23_next_0_months", 2).alias("is_active_sum_past_23_next_0_months")),
    f.round(f.median("month_distinct_skus_sold_mean_past_11_next_0_months", 2).alias("month_distinct_skus_sold_mean_past_11_next_0_months")),
    f.round(f.median("month_distinct_product_categories_mean_past_11_next_0_months", 2).alias("month_distinct_product_categories_mean_past_11_next_0_months")),
]

In [12]:
agg_cols

[Column<'round(median(most_trx_branch_n_places_within_0300m__All), 2) AS most_trx_branch_n_places_within_0300m__All'>,
 Column<'round(median(most_trx_branch_pop_density), 2) AS most_trx_branch_pop_density'>,
 Column<'round(median(most_frequent_day_of_month_mean_past_11_next_0_months), 2) AS most_frequent_day_of_month_mean_past_11_next_0_months'>,
 Column<'round(median(most_frequent_day_of_week_mean_past_11_next_0_months), 2) AS most_frequent_day_of_week_mean_past_11_next_0_months'>,
 Column<'round(median(last_trx_net_sales), 2) AS last_trx_net_sales'>,
 Column<'round(median(last_trx_total_discount_percentual), 2) AS last_trx_total_discount_percentual'>,
 Column<'round(median(last_trx_current_mileage), 2) AS last_trx_current_mileage'>,
 Column<'round(median(month_avg_mileage_per_day_nullif_past_11_next_0_months), 2) AS month_avg_mileage_per_day_nullif_past_11_next_0_months'>,
 Column<'round(median(customer_synthetic_oil), 2) AS customer_synthetic_oil'>,
 Column<'round(median(month_avg_o

In [9]:
segmentation_df.filter(
    f.col("_observ_end_dt") >= "2021-01-01"
).filter(
    f.col("_observ_end_dt") < "2025-01-01"
).filter(
    f.col("is_active") > 0
).groupBy("loyalty_segment", "cluster_id").agg(
    *agg_cols
).orderBy("loyalty_segment", "cluster_id").show(truncate=False)

+---------------+----------+------------------------------------------+---------------------------+-----------------------------------------------------+----------------------------------------------------+------------------+----------------------------------+------------------------+------------------------------------------------------+----------------------+------------------------------------------+-----------------------------------------+-----------------------------------------------------+----------------------------------------------------+-----------------------------------+---------------------------------------------------+------------------------------------------------------------+
|loyalty_segment|cluster_id|most_trx_branch_n_places_within_0300m__All|most_trx_branch_pop_density|most_frequent_day_of_month_mean_past_11_next_0_months|most_frequent_day_of_week_mean_past_11_next_0_months|last_trx_net_sales|last_trx_total_discount_percentual|last_trx_current_mileage|month_avg_

In [11]:
segmentation_df.filter(
    f.col("_observ_end_dt") >= "2021-01-01"
).filter(
    f.col("_observ_end_dt") < "2025-01-01"
).filter(
    f.col("is_active") > 0
).groupBy("loyalty_segment").agg(
    *agg_cols
).orderBy("loyalty_segment").show(truncate=False)

segmentation_df.filter(
    f.col("_observ_end_dt") >= "2021-01-01"
).filter(
    f.col("_observ_end_dt") < "2025-01-01"
).filter(
    f.col("is_active") > 0
).select(
    *agg_cols
).show(truncate=False)

+---------------+------------------------------------------+---------------------------+-----------------------------------------------------+----------------------------------------------------+------------------+----------------------------------+------------------------+------------------------------------------------------+----------------------+------------------------------------------+-----------------------------------------+-----------------------------------------------------+----------------------------------------------------+-----------------------------------+---------------------------------------------------+------------------------------------------------------------+
|loyalty_segment|most_trx_branch_n_places_within_0300m__All|most_trx_branch_pop_density|most_frequent_day_of_month_mean_past_11_next_0_months|most_frequent_day_of_week_mean_past_11_next_0_months|last_trx_net_sales|last_trx_total_discount_percentual|last_trx_current_mileage|month_avg_mileage_per_day_nullif

+------------------------------------------+---------------------------+-----------------------------------------------------+----------------------------------------------------+------------------+----------------------------------+------------------------+------------------------------------------------------+----------------------+------------------------------------------+-----------------------------------------+-----------------------------------------------------+----------------------------------------------------+-----------------------------------+---------------------------------------------------+------------------------------------------------------------+
|most_trx_branch_n_places_within_0300m__All|most_trx_branch_pop_density|most_frequent_day_of_month_mean_past_11_next_0_months|most_frequent_day_of_week_mean_past_11_next_0_months|last_trx_net_sales|last_trx_total_discount_percentual|last_trx_current_mileage|month_avg_mileage_per_day_nullif_past_11_next_0_months|customer_